## Load the Model

In [2]:
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
import wandb
import gradio as gr

root = Path('.')  # notebook is in TREES/
images_dir = root / "images"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

c:\Users\rawil\anaconda3\envs\SoPa\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [ ]:
model_path = "models/Resnet18_V3.pth"

model = models.resnet18(weights=None)  # don't load pretrained weights
model.fc = nn.Linear(model.fc.in_features, 5)
state_dict = torch.load(model_path, map_location=torch.device(device))
model.load_state_dict(state_dict)
model.eval()

class_names = ["koivu", "kuusi", "lehmus", "pihlaja", "vaahtera"]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

def predict(image: Image.Image):
    """
    Takes a PIL image, transforms it, and returns a dictionary of class confidences in %.
    """
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)[0]
        # Convert to percentages
        confidences = {class_names[i]: float(probabilities[i] * 100) for i in range(len(class_names))}

    return confidences

In [ ]:
examples = [
    ['lehmus/lehmus6.jpg'],
    ['pihlaja/pihlaja23.jpg'],
    ['koivu/koivu15.jpg']
]
examples = [[os.path.join(images_dir, path[0])] for path in examples]

iface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Label(num_top_classes=3, label="Predictions"),
    title="5-Class Image Classifier",
    description="Upload an image to see the model's prediction.",
    examples=examples
)

iface.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2025/10/03 12:23:26 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout
